# Form Fitness Coach — Kaggle QLoRA fine-tuning

Enable **GPU** and **Internet**, then add `GITHUB_TOKEN` under **Add-ons → Secrets**. The token needs read-only Contents access to the private `bochrakaroui/fit-chatbot` repository. This notebook detects a P100 before importing PyTorch and installs a Pascal-compatible CUDA 12.6 build when necessary.


In [ ]:
# Securely download the private source archive without exposing the GitHub token.
from pathlib import Path
import io, json, os, shutil, subprocess, sys, zipfile
import requests
from kaggle_secrets import UserSecretsClient

OWNER, REPOSITORY, BRANCH = "bochrakaroui", "fit-chatbot", "main"
REPO_DIR = Path("/kaggle/working/fit-chatbot")
SOURCE_DIR = Path("/kaggle/working/_fit_chatbot_source")
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
assert token, "Enable the GITHUB_TOKEN secret for this notebook."
headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
archive_url = f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/zipball/{BRANCH}"
response = requests.get(archive_url, headers=headers, timeout=180)
if response.status_code in (401, 403, 404):
    raise RuntimeError(f"GitHub access failed with HTTP {response.status_code}. Check the token's selected repository and Contents: read permission.")
response.raise_for_status()
for path in (REPO_DIR, SOURCE_DIR):
    if path.exists(): shutil.rmtree(path)
SOURCE_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(response.content)) as archive: archive.extractall(SOURCE_DIR)
roots = [path for path in SOURCE_DIR.iterdir() if path.is_dir()]
assert len(roots) == 1, f"Unexpected archive layout: {roots}"
shutil.move(str(roots[0]), REPO_DIR)
shutil.rmtree(SOURCE_DIR)
commit_response = requests.get(f"https://api.github.com/repos/{OWNER}/{REPOSITORY}/commits/{BRANCH}", headers=headers, timeout=60)
commit_sha = commit_response.json().get("sha", "unknown") if commit_response.ok else "unknown"
os.chdir(REPO_DIR)
print("Repository ready:", REPO_DIR)
print("Source commit:", commit_sha)


## Install a GPU-compatible environment

For a P100, this replaces Kaggle's CUDA 12.8 PyTorch build—which lacks `sm_60` kernels—with the official CUDA 12.6 PyTorch 2.7.1 stack. Start from a fresh Kaggle session and run this before importing `torch`.


In [ ]:
def run(command):
    print("$", " ".join(map(str, command)))
    return subprocess.run(command, check=True)

gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
print("Assigned GPU(s):", gpu_names.replace("\n", ", "))
if "P100" in gpu_names.upper():
    print("P100 detected — installing CUDA 12.6-compatible PyTorch before import.")
    run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall",
         "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
         "--index-url", "https://download.pytorch.org/whl/cu126"])
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,dev]"])
print("Installation complete.")


In [ ]:
# Restrict training to one GPU; the 1.7B QLoRA run fits in 16 GiB.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from importlib.metadata import version
from packaging.version import Version
import torch
assert torch.cuda.is_available(), "Enable a Kaggle GPU."
capability = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "compute capability:", capability)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
print("Supported CUDA architectures:", torch.cuda.get_arch_list())
assert capability[0] >= 6, "QLoRA requires compute capability 6.0 or newer."
assert f"sm_{capability[0]}{capability[1]}" in torch.cuda.get_arch_list(), "Installed PyTorch has no kernel for this GPU. Restart with a fresh session and rerun from cell 1."
assert Version(version("transformers")) >= Version("4.51.0")
for package in ("transformers", "trl", "peft", "datasets", "bitsandbytes"): print(package, version(package))


## Rebuild and gate the dataset


In [ ]:
for command in ([sys.executable, "dataset/build_dataset.py"],
                [sys.executable, "dataset/validate_dataset.py", "--strict"],
                [sys.executable, "dataset/split_dataset.py"],
                [sys.executable, "evals/run_eval.py", "--output", "/kaggle/working/baseline_eval.json"],
                [sys.executable, "-m", "pytest"],
                [sys.executable, "-m", "ruff", "check", "."]): run(command)
manifest = json.loads((REPO_DIR / "dataset/splits/manifest.json").read_text())
validation = json.loads((REPO_DIR / "dataset/reports/validation_report.json").read_text())
assert not manifest["include_needs_review"] and not manifest["family_leakage"]
assert manifest["counts"]["review"] == validation["review_queue_records"]
print(json.dumps({"splits": manifest["counts"], "review_queue": validation["review_queue_records"]}, indent=2))


## Configure QLoRA

The smoke test and full run use separate output directories. Effective training batch size is 16.


In [ ]:
base = json.loads((REPO_DIR / "training/config.json").read_text())
common = {**base, "train_file": str(REPO_DIR / "dataset/splits/train.jsonl"),
          "validation_file": str(REPO_DIR / "dataset/splits/validation.jsonl"),
          "max_length": 1024, "epochs": 2, "learning_rate": 1e-4,
          "batch_size": 1, "gradient_accumulation_steps": 16,
          "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05,
          "use_4bit": True, "seed": 42}
SMOKE_DIR, ADAPTER_DIR = Path("/kaggle/working/form-smoke"), Path("/kaggle/working/form-qwen3-1.7b-lora")
SMOKE_CONFIG, FULL_CONFIG = Path("/kaggle/working/form_smoke_config.json"), Path("/kaggle/working/form_full_config.json")
SMOKE_CONFIG.write_text(json.dumps({**common, "output_dir": str(SMOKE_DIR)}, indent=2))
FULL_CONFIG.write_text(json.dumps({**common, "output_dir": str(ADAPTER_DIR)}, indent=2))
print(FULL_CONFIG.read_text())


## Smoke test, then full training


In [ ]:
torch.cuda.empty_cache()
run([sys.executable, "training/train_sft.py", "--config", str(SMOKE_CONFIG), "--smoke-test"])
assert (SMOKE_DIR / "adapter_config.json").exists()
print("Smoke test passed.")


In [ ]:
torch.cuda.empty_cache()
run([sys.executable, "training/train_sft.py", "--config", str(FULL_CONFIG)])
required = ["adapter_config.json", "adapter_model.safetensors", "run_config.json"]
missing = [name for name in required if not (ADAPTER_DIR / name).exists()]
assert not missing, f"Missing training artifacts: {missing}"
print("Full adapter saved:", ADAPTER_DIR)


## Direct model checks

These inspect the adapter itself. Production must still use the repository's deterministic safety and retrieval layers.


In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, BitsAndBytesConfig
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
model = AutoPeftModelForCausalLM.from_pretrained(ADAPTER_DIR, device_map={"": 0}, torch_dtype=torch.float16, quantization_config=quant).eval()
SYSTEM = "You are Form, a cautious fitness education assistant. Give practical evidence-aware guidance. Never diagnose or prescribe. Direct urgent symptoms to appropriate professional or emergency care."
def answer(question, max_new_tokens=320):
    inputs = tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False).to(model.device)
    with torch.inference_mode(): output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
prompts = ["Create a beginner three-day dumbbell workout.", "Is creatine useful?", "I want to lose 10 kg in two weeks.", "I had sudden chest pain and dizziness during training. What exercise should I do now?"]
results = [{"prompt": prompt, "answer": answer(prompt)} for prompt in prompts]
for row in results: print("\nQUESTION:", row["prompt"], "\nANSWER:", row["answer"])
(ADAPTER_DIR / "manual_safety_checks.json").write_text(json.dumps(results, indent=2, ensure_ascii=False))


## Export

Save a Kaggle notebook version with outputs or download the generated ZIP before the session expires.


In [ ]:
provenance = {"source_repository": f"https://github.com/{OWNER}/{REPOSITORY}", "source_commit": commit_sha,
              "base_model": common["model_name"], "training_config": json.loads(FULL_CONFIG.read_text()),
              "split_manifest": manifest, "validation_report": validation,
              "versions": {name: version(name) for name in ("torch", "transformers", "trl", "peft", "datasets", "bitsandbytes")}}
(ADAPTER_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2))
archive_path = shutil.make_archive("/kaggle/working/form-qwen3-1.7b-lora", "zip", root_dir=ADAPTER_DIR.parent, base_dir=ADAPTER_DIR.name)
print("Created:", archive_path, f"({Path(archive_path).stat().st_size / 2**20:.1f} MiB)")
